# Тестирование Spark подключения

Этот notebook проверяет работоспособность Spark кластера и тестирует загрузку данных.


In [2]:
# Импорт библиотек
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os


In [3]:
# Создание Spark сессии в local mode (без кластера)
spark = SparkSession.builder \
    .appName("ETL_Test_Local") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark версия: {spark.version}")
print(f"Spark контекст: {spark.sparkContext.appName}")
print(f"Master URL: {spark.sparkContext.master}")
print("🎉 Spark запущен в local mode - все операции будут выполняться в этом контейнере")


Spark версия: 3.5.0
Spark контекст: ETL_Test_Local
Master URL: local[*]
🎉 Spark запущен в local mode - все операции будут выполняться в этом контейнере


In [4]:
# Тест 1: Простая операция с данными
test_data = [(1, "Alice", 25), (2, "Bob", 30), (3, "Charlie", 35)]
test_df = spark.createDataFrame(test_data, ["id", "name", "age"])

print("Тестовые данные:")
test_df.show()

print(f"Количество строк: {test_df.count()}")
print(f"Количество партиций: {test_df.rdd.getNumPartitions()}")


Тестовые данные:
+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 35|
+---+-------+---+

Количество строк: 3
Количество партиций: 12


In [5]:
# Тест 2: Чтение CSV файлов из проекта  
csv_files = [
    "/home/jovyan/data/tables/md_currency_d.csv",
    "/home/jovyan/data/tables/md_account_d.csv"
]

for file_path in csv_files:
    if os.path.exists(file_path):
        print(f"\nЧтение файла: {file_path}")
        
        df = spark.read.options(
            header="true",
            inferSchema="true", 
            sep=";"
        ).csv(file_path)
        
        print(f"Строк: {df.count()}")
        print(f"Колонок: {len(df.columns)}")
        print("Первые 3 строки:")
        df.show(3)
    else:
        print(f"Файл не найден: {file_path}")



Чтение файла: /home/jovyan/data/tables/md_currency_d.csv
Строк: 50
Колонок: 5
Первые 3 строки:
+-----------+----------------+--------------------+-------------+-------------+
|CURRENCY_RK|DATA_ACTUAL_DATE|DATA_ACTUAL_END_DATE|CURRENCY_CODE|CODE_ISO_CHAR|
+-----------+----------------+--------------------+-------------+-------------+
|    4586704|      2011-09-06|          2050-12-31|            0|          NON|
|         50|      2017-05-11|          2050-12-31|          356|          INR|
|         51|      2017-05-11|          2050-12-31|          484|          MXN|
+-----------+----------------+--------------------+-------------+-------------+
only showing top 3 rows


Чтение файла: /home/jovyan/data/tables/md_account_d.csv
Строк: 112
Колонок: 7
Первые 3 строки:
+----------------+--------------------+----------+--------------------+---------+-----------+-------------+
|DATA_ACTUAL_DATE|DATA_ACTUAL_END_DATE|ACCOUNT_RK|      ACCOUNT_NUMBER|CHAR_TYPE|CURRENCY_RK|CURRENCY_CODE|
+------

In [6]:
# Тест 3: Информация о кластере и завершение
print("Информация о Spark кластере:")

# Безопасное получение базовой информации
try:
    app_id = spark.sparkContext.applicationId
    print(f"Приложение ID: {app_id}")
except Exception as e:
    print(f"Не удалось получить Application ID: {e}")

try:
    ui_url = getattr(spark.sparkContext, "uiWebUrl", None)
    ui_url = ui_url if ui_url else "недоступен (Spark UI отключен или порт закрыт)"
    print(f"UI URL: {ui_url}")
except Exception as e:
    print(f"Не удалось получить UI URL: {e}")

# Информация о конфигурации Spark
try:
    print(f"Режим выполнения: {spark.sparkContext.master}")
    print(f"Количество CPU: {spark.sparkContext.defaultParallelism}")
except Exception as e:
    print(f"Не удалось получить конфигурацию: {e}")

# Попробуем получить информацию об исполнителях (может не работать в local mode)
try:
    status_tracker = spark.sparkContext.statusTracker()
    get_exec_infos = getattr(status_tracker, 'getExecutorInfos', None)
    if callable(get_exec_infos):
        executors = get_exec_infos()
        print(f"Количество исполнителей: {len(executors)}")
    else:
        print("В local mode информация об исполнителях недоступна")
except Exception as e:
    print(f"Не удалось получить информацию об исполнителях: {e}")

print("\n✅ Все тесты Spark подключения завершены!")
print("🚀 Теперь можно переходить к тестированию гибридного DAG в Airflow!")
try:
    print("📊 Spark UI доступен по адресу:", getattr(spark.sparkContext, "uiWebUrl", None))
except Exception as e:
    print(f"Не удалось вывести адрес Spark UI: {e}")


Информация о Spark кластере:
Приложение ID: local-1754656115367
UI URL: http://cd08d9ef2c40:4040
Режим выполнения: local[*]
Количество CPU: 12
В local mode информация об исполнителях недоступна

✅ Все тесты Spark подключения завершены!
🚀 Теперь можно переходить к тестированию гибридного DAG в Airflow!
📊 Spark UI доступен по адресу: http://cd08d9ef2c40:4040
